In [3]:
import pandas as pd
import re

df = pd.read_csv("clean_job_postings.csv")
df["full_text"] = df["full_text"].fillna("")

# --- Clue 1: Urgency language ---
urgency_words = ["apply now", "urgent", "immediate start", "limited time",
                  "act fast", "hurry", "don't miss"]

def count_urgency(text):
    text = text.lower()
    return sum(text.count(word) for word in urgency_words)

df["urgency_score"] = df["full_text"].apply(count_urgency)

# --- Clue 2: Vagueness score (count how many key fields are missing/blank) ---
def vagueness_score(row):
    score = 0
    if pd.isna(row["company_profile"]) or str(row["company_profile"]).strip() == "":
        score += 1
    if pd.isna(row["required_education"]) or str(row["required_education"]).strip() == "":
        score += 1
    if pd.isna(row["industry"]) or str(row["industry"]).strip() == "":
        score += 1
    if pd.isna(row["benefits"]) or str(row["benefits"]).strip() == "":
        score += 1
    return score

df["vagueness_score"] = df.apply(vagueness_score, axis=1)

# --- Clue 3: Text length ---
df["text_length"] = df["full_text"].apply(lambda x: len(x.split()))

# --- Clue 4: Exclamation / all-caps intensity ---
def caps_and_punctuation_score(text):
    exclamations = text.count("!")
    caps_words = len(re.findall(r"\b[A-Z]{3,}\b", text))
    return exclamations + caps_words

df["hype_score"] = df["full_text"].apply(caps_and_punctuation_score)

# Keep the structured flags from the raw data too
final_cols = ["full_text", "urgency_score", "vagueness_score", "text_length",
              "hype_score", "telecommuting", "has_company_logo",
              "has_questions", "fraudulent"]

df_final = df[final_cols]
df_final.to_csv("featured_job_postings.csv", index=False)
print("Saved: featured_job_postings.csv")
print(df_final.head())

Saved: featured_job_postings.csv
                                           full_text  urgency_score  \
0  Marketing Intern We're Food52, and we've creat...              0   
1  Customer Service - Cloud Video Production 90 S...              0   
2  Commissioning Machinery Assistant (CMA) Valor ...              0   
3  Account Executive - Washington DC Our passion ...              0   
4  Bill Review Manager SpotSource Solutions LLC i...              0   

   vagueness_score  text_length  hype_score  telecommuting  has_company_logo  \
0                3          382           3              0                 1   
1                1          901           7              0                 1   
2                3          359           1              0                 1   
3                0          709          12              0                 1   
4                0          470          16              0                 1   

   has_questions  fraudulent  
0              0           0